In [1]:
import numpy as np
import pandas as pd
import plotly.express as px


# =========================
# Configuration
# =========================
RANDOM_SEED = 42
N_MONTHS = 12
INITIAL_CUSTOMERS = 4000
MONTHLY_NEW_CUSTOMERS = 250

MONTH_START = "2025-01-01"

SEGMENT_LABELS = ["Group 1", "Group 2", "Group 3", "Group 4", "Group 5"]
MIGRATION_LABELS = ["Stayed", "Improved", "Worsened", "New", "Exited"]

np.random.seed(RANDOM_SEED)


# =========================
# Helpers
# =========================
def generate_month_sequence(start_date: str, n_months: int) -> pd.DatetimeIndex:
    """Generate monthly period starts."""
    return pd.date_range(start=start_date, periods=n_months, freq="MS")


def score_to_group(score: pd.Series) -> pd.Series:
    """
    Convert score 0-100 into 5 groups.
    Higher score => better group.
    Group 1: 0-20
    Group 2: 21-40
    Group 3: 41-60
    Group 4: 61-80
    Group 5: 81-100
    """
    bins = [-0.001, 20, 40, 60, 80, 100]
    return pd.cut(score, bins=bins, labels=SEGMENT_LABELS, include_lowest=True)


def initialize_portfolio(n_customers: int) -> pd.DataFrame:
    """Create initial customer base with synthetic scores."""
    customer_ids = [f"CUST_{i:06d}" for i in range(1, n_customers + 1)]

    # Mixture to create more realistic scoring distribution
    base_scores = np.concatenate([
        np.random.normal(loc=30, scale=10, size=n_customers // 3),
        np.random.normal(loc=55, scale=12, size=n_customers // 3),
        np.random.normal(loc=78, scale=10, size=n_customers - 2 * (n_customers // 3))
    ])

    base_scores = np.clip(base_scores, 0, 100).round(0)

    df = pd.DataFrame({
        "customer_id": customer_ids,
        "score": base_scores
    })

    df["score_group"] = score_to_group(df["score"])
    df["is_active"] = True
    return df


def apply_monthly_transition(active_df: pd.DataFrame) -> pd.DataFrame:
    """
    Simulate score movement and exits for active customers.
    Some improve, some worsen, some stay relatively stable.
    """
    df = active_df.copy()

    n = len(df)

    # Exit flag: some customers leave the portfolio
    exit_prob = np.random.uniform(0.03, 0.07)
    df["is_active_next"] = np.random.rand(n) > exit_prob

    # Score movement: mostly small variations, some strong shifts
    noise = np.random.normal(loc=0, scale=8, size=n)
    shock_flag = np.random.rand(n) < 0.10
    shock = np.random.normal(loc=0, scale=18, size=n)

    score_change = noise + shock_flag * shock

    # Add slight mean reversion toward middle ranges
    mean_reversion = -(df["score"] - 55) * 0.05
    df["next_score"] = np.clip(df["score"] + score_change + mean_reversion, 0, 100).round(0)

    # Exited customers do not appear next month
    next_active_df = df.loc[df["is_active_next"], ["customer_id", "next_score"]].copy()
    next_active_df = next_active_df.rename(columns={"next_score": "score"})
    next_active_df["score_group"] = score_to_group(next_active_df["score"])
    next_active_df["is_active"] = True

    return next_active_df


def generate_new_customers(start_index: int, n_new_customers: int) -> pd.DataFrame:
    """Generate new customers entering the portfolio."""
    customer_ids = [f"CUST_{i:06d}" for i in range(start_index, start_index + n_new_customers)]

    # New customers centered more in middle-lower range
    new_scores = np.random.normal(loc=50, scale=15, size=n_new_customers)
    new_scores = np.clip(new_scores, 0, 100).round(0)

    df = pd.DataFrame({
        "customer_id": customer_ids,
        "score": new_scores
    })
    df["score_group"] = score_to_group(df["score"])
    df["is_active"] = True
    return df


def classify_migration(previous_group: str, current_group: str) -> str:
    """
    Migration logic:
    - New: only in current month
    - Exited: only in previous month
    - Stayed: same group
    - Improved: current group is better than previous group
    - Worsened: current group is worse than previous group
    """
    if pd.isna(previous_group) and pd.notna(current_group):
        return "New"
    if pd.notna(previous_group) and pd.isna(current_group):
        return "Exited"
    if previous_group == current_group:
        return "Stayed"

    prev_idx = SEGMENT_LABELS.index(previous_group)
    curr_idx = SEGMENT_LABELS.index(current_group)

    if curr_idx > prev_idx:
        return "Improved"
    return "Worsened"


# =========================
# Data generation
# =========================
months = generate_month_sequence(MONTH_START, N_MONTHS)

monthly_snapshots = []
portfolio_df = initialize_portfolio(INITIAL_CUSTOMERS)
next_customer_index = INITIAL_CUSTOMERS + 1

for month in months:
    snapshot = portfolio_df.copy()
    snapshot["month"] = month
    monthly_snapshots.append(snapshot)

    # Prepare next month portfolio
    next_portfolio = apply_monthly_transition(portfolio_df)

    new_customers_df = generate_new_customers(
        start_index=next_customer_index,
        n_new_customers=MONTHLY_NEW_CUSTOMERS
    )
    next_customer_index += MONTHLY_NEW_CUSTOMERS

    portfolio_df = pd.concat([next_portfolio, new_customers_df], ignore_index=True)


score_history_df = pd.concat(monthly_snapshots, ignore_index=True)
score_history_df["month_str"] = score_history_df["month"].dt.strftime("%Y-%m")


# =========================
# Monthly stability
# =========================
monthly_stability_df = (
    score_history_df
    .groupby(["month", "score_group"], observed=False)
    .agg(customer_count=("customer_id", "nunique"))
    .reset_index()
)

monthly_totals_df = (
    monthly_stability_df
    .groupby("month", as_index=False)
    .agg(total_customers=("customer_count", "sum"))
)

monthly_stability_df = monthly_stability_df.merge(monthly_totals_df, on="month", how="left")
monthly_stability_df["share_pct"] = (
    monthly_stability_df["customer_count"] / monthly_stability_df["total_customers"] * 100
).round(2)
monthly_stability_df["month_str"] = monthly_stability_df["month"].dt.strftime("%Y-%m")


# =========================
# Monthly migration
# =========================
migration_records = []

for i in range(1, len(months)):
    prev_month = months[i - 1]
    curr_month = months[i]

    prev_df = (
        score_history_df.loc[score_history_df["month"] == prev_month, ["customer_id", "score_group"]]
        .rename(columns={"score_group": "previous_group"})
    )

    curr_df = (
        score_history_df.loc[score_history_df["month"] == curr_month, ["customer_id", "score_group"]]
        .rename(columns={"score_group": "current_group"})
    )

    merged_df = prev_df.merge(curr_df, on="customer_id", how="outer")

    merged_df["migration_status"] = merged_df.apply(
        lambda row: classify_migration(row["previous_group"], row["current_group"]),
        axis=1
    )

    merged_df["from_month"] = prev_month
    merged_df["to_month"] = curr_month

    # Optional origin/destination group columns for analysis
    merged_df["from_group"] = merged_df["previous_group"].astype("object")
    merged_df["to_group"] = merged_df["current_group"].astype("object")

    migration_records.append(merged_df)

monthly_migration_detail_df = pd.concat(migration_records, ignore_index=True)

monthly_migration_summary_df = (
    monthly_migration_detail_df
    .groupby(["to_month", "migration_status"], as_index=False)
    .agg(customer_count=("customer_id", "nunique"))
)

monthly_migration_totals_df = (
    monthly_migration_summary_df
    .groupby("to_month", as_index=False)
    .agg(total_customers=("customer_count", "sum"))
)

monthly_migration_summary_df = monthly_migration_summary_df.merge(
    monthly_migration_totals_df,
    on="to_month",
    how="left"
)

monthly_migration_summary_df["share_pct"] = (
    monthly_migration_summary_df["customer_count"] /
    monthly_migration_summary_df["total_customers"] * 100
).round(2)

monthly_migration_summary_df["month_str"] = monthly_migration_summary_df["to_month"].dt.strftime("%Y-%m")


# =========================
# Transition matrix by month
# =========================
transition_matrix_df = (
    monthly_migration_detail_df.loc[
        monthly_migration_detail_df["migration_status"].isin(["Stayed", "Improved", "Worsened"])
    ]
    .groupby(["to_month", "from_group", "to_group"], as_index=False)
    .agg(customer_count=("customer_id", "nunique"))
)

transition_matrix_df["month_str"] = transition_matrix_df["to_month"].dt.strftime("%Y-%m")


# =========================
# Example charts with Plotly Express
# =========================

# 1) Monthly stability by score group
fig_stability = px.line(
    monthly_stability_df,
    x="month",
    y="share_pct",
    color="score_group",
    markers=True,
    title="Monthly Stability of Score Groups",
    labels={
        "month": "Month",
        "share_pct": "Portfolio Share (%)",
        "score_group": "Score Group"
    }
)

# 2) Monthly customer count by score group
fig_stability_count = px.area(
    monthly_stability_df,
    x="month",
    y="customer_count",
    color="score_group",
    title="Monthly Customer Count by Score Group",
    labels={
        "month": "Month",
        "customer_count": "Customers",
        "score_group": "Score Group"
    }
)

# 3) Monthly migration status
fig_migration = px.bar(
    monthly_migration_summary_df,
    x="to_month",
    y="share_pct",
    color="migration_status",
    barmode="stack",
    category_orders={"migration_status": MIGRATION_LABELS},
    title="Monthly Migration Status Share",
    labels={
        "to_month": "Month",
        "share_pct": "Share (%)",
        "migration_status": "Migration Status"
    }
)

# 4) Transition heatmap for latest month
latest_month = transition_matrix_df["to_month"].max()
latest_transition_df = transition_matrix_df.loc[
    transition_matrix_df["to_month"] == latest_month
].copy()

fig_transition_heatmap = px.density_heatmap(
    latest_transition_df,
    x="from_group",
    y="to_group",
    z="customer_count",
    histfunc="sum",
    title=f"Transition Matrix - {latest_month.strftime('%Y-%m')}",
    labels={
        "from_group": "Previous Group",
        "to_group": "Current Group",
        "customer_count": "Customers"
    }
)


# =========================
# Output preview
# =========================
print("=== Score History Sample ===")
print(score_history_df.head())

print("\n=== Monthly Stability Sample ===")
print(monthly_stability_df.head(10))

print("\n=== Monthly Migration Summary Sample ===")
print(monthly_migration_summary_df.head(10))

print("\n=== Transition Matrix Sample ===")
print(transition_matrix_df.head(10))

=== Score History Sample ===
   customer_id  score score_group  is_active      month month_str
0  CUST_000001   35.0     Group 2       True 2025-01-01   2025-01
1  CUST_000002   29.0     Group 2       True 2025-01-01   2025-01
2  CUST_000003   36.0     Group 2       True 2025-01-01   2025-01
3  CUST_000004   45.0     Group 3       True 2025-01-01   2025-01
4  CUST_000005   28.0     Group 2       True 2025-01-01   2025-01

=== Monthly Stability Sample ===
       month score_group  customer_count  total_customers  share_pct month_str
0 2025-01-01     Group 1             217             4000       5.42   2025-01
1 2025-01-01     Group 2            1054             4000      26.35   2025-01
2 2025-01-01     Group 3            1006             4000      25.15   2025-01
3 2025-01-01     Group 4            1164             4000      29.10   2025-01
4 2025-01-01     Group 5             559             4000      13.98   2025-01
5 2025-02-01     Group 1             280             4056       6.9

In [2]:
# Uncomment to display charts in a notebook or script environment
fig_stability.show()
fig_stability_count.show()
fig_migration.show()
fig_transition_heatmap.show()